<a href="https://colab.research.google.com/github/NoelAR1103/in-silico-D-T-P/blob/main/D1DTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install biopython
import Bio
print("setup successfull! Biopython version:", Bio.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.1 MB/s  0:00:00
setup successfull! Biopython version: 1.88


In [3]:
# Our single target pathogen
pathogen = "Acinetobacter baumannii"

# A real Beta-lactamase resistance protein sequence from A. baumannii
protein_seq = "MRVLALSAVFLVASIIGMPAVAKEWQENKSWNAHFTEHKSQGVVVLWNENKQQGFTNNLKRANQAFLPASTFKIPNSLIALDLGVVKDEHQVFKWDGQTRDIATWNRDHNLITAMKYSVVPVYQEFARQIGEARMSKMLHAFDYGNEDISGNVDSFWLDGGIRISATEQISFLRKLYHNKLHVSERSQRI"

# Let's inspect the protein
print("Target:", pathogen)
print("Length (amino acids):", len(protein_seq))
print("Cysteine count:", protein_seq.count("C"))
print("Alanine count:", protein_seq.count("A"))

Target: Acinetobacter baumannii
Length (amino acids): 190
Cysteine count: 0
Alanine count: 16


In [4]:
import urllib.request
from Bio import SeqIO

# 1. Download a real A. baumannii carbapenemase protein (OXA-23) in FASTA format
uniprot_id = "Q9L4P2"
url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
urllib.request.urlretrieve(url, "oxa23.fasta")

# 2. Use Biopython to read the downloaded FASTA file
record = SeqIO.read("oxa23.fasta", "fasta")

print("Protein ID:", record.id)
print("Description:", record.description)
print("Full Length:", len(record.seq))
print("First 30 amino acids:", record.seq[:30])

Protein ID: sp|Q9L4P2|BLO23_ACIBA
Description: sp|Q9L4P2|BLO23_ACIBA Beta-lactamase OXA-23 OS=Acinetobacter baumannii OX=470 GN=OXA-23 PE=1 SV=1
Full Length: 273
First 30 amino acids: MNKYFTCYVVASLFLSGCTVQHNLINETPS


In [5]:
import urllib.request
from Bio import SeqIO

# 1. Download ALL reviewed (Swiss-Prot) proteins for Acinetobacter baumannii (TaxID: 470)
url = "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=(taxonomy_id:470)+AND+(reviewed:true)"
urllib.request.urlretrieve(url, "abaumannii_reviewed.fasta")

# 2. Parse all records into a Python list
all_proteins = list(SeqIO.parse("abaumannii_reviewed.fasta", "fasta"))
print("Total reviewed A. baumannii proteins downloaded:", len(all_proteins))

# 3. FILTER #1: Keep only proteins >= 100 amino acids (capable of forming stable drug-binding pockets)
filtered_proteins = []

for protein in all_proteins:
    if len(protein.seq) >= 100:
        filtered_proteins.append(protein)

print("Proteins passing Length Filter (>= 100 aa):", len(filtered_proteins))
print("Proteins eliminated (too small):", len(all_proteins) - len(filtered_proteins))

Total reviewed A. baumannii proteins downloaded: 1845
Proteins passing Length Filter (>= 100 aa): 1676
Proteins eliminated (too small): 169


In [6]:
from Bio import SeqIO

# 1. Remove exact duplicate sequences (Non-redundant proteome)
seen_sequences = set()
unique_proteins = []

for protein in filtered_proteins:
    seq_str = str(protein.seq)
    if seq_str not in seen_sequences:
        seen_sequences.add(seq_str)
        unique_proteins.append(protein)

print("Proteins before deduplication:", len(filtered_proteins))
print("Unique non-redundant proteins:", len(unique_proteins))
print("Duplicates removed:", len(filtered_proteins) - len(unique_proteins))

# 2. Save our surviving candidates to a new FASTA file for the next stage
SeqIO.write(unique_proteins, "filter2_unique.fasta", "fasta")
print("Saved successfully to filter2_unique.fasta!")

Proteins before deduplication: 1676
Unique non-redundant proteins: 766
Duplicates removed: 910
Saved successfully to filter2_unique.fasta!
